# Task A — Data Understanding and Limitations

## 1. Competition and Analysis Context

This notebook documents Task A for the NHPA healthcare-claim fraud-risk competition. `label = 1` represents a claim classified as fraud under the competition definition, while `label = 0` represents non-fraud. A later model will produce `fraud_probability` values that rank claims for human investigation; NHPA can audit only the highest-risk 5% of claims.

This notebook does not train a fraud model, clean the production dataset, or establish causal relationships. Differences in observed fraud prevalence are statistical associations in this training data and may reflect confounding, provider mix, clinical complexity, geography, coding practices, data quality, or unobserved factors. `claim_id` is an observation identifier only and must never be used as a predictive feature or a source of engineered features.

## 2. Imports and Configuration

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from scipy import stats

RANDOM_STATE = 42
MIN_GROUP_SIZE = 100
CARDINALITY_BINS = [10, 50, 200]
NUMERIC_QUANTILES = [0, 0.001, 0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99, 0.999, 1.0]
PLACEHOLDER_VALUES = {"", "-", "na", "n/a", "unknown", "null", "none"}

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_rows", 30)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")
plt.rcParams.update({"figure.figsize": (10, 5), "axes.grid": True, "grid.alpha": 0.25})


def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("Could not locate pyproject.toml from the current directory.")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
DATA_PATH = PROJECT_ROOT / "data" / "train.csv"

print("Python:", sys.version.split()[0])
print("pandas:", pd.__version__)
print("numpy:", np.__version__)
print("Training data:", DATA_PATH)

## 3. Load Data

The raw training file is read without row removal, imputation, or value normalization. Code-like fields are explicitly loaded as strings so leading zeros remain intact.

In [ ]:
code_like_columns = [
    "claim_id",
    "kdkc",
    "dati2",
    "typeppk",
    "jkpst",
    "jnspelsep",
    "cmg",
    "severitylevel",
    "diagprimer",
]

dtype_overrides = {column: "string" for column in code_like_columns}
train = pd.read_csv(DATA_PATH, dtype=dtype_overrides)

assert "claim_id" in train.columns
assert "label" in train.columns
assert set(train["label"].dropna().unique()).issubset({0, 1})

print(f"Rows: {len(train):,}")
print(f"Columns: {train.shape[1]:,}")
print(f"Training file: {DATA_PATH.relative_to(PROJECT_ROOT)}")

## 4. Initial Dataset Overview

The overview is intentionally compact: it exposes schema and representative rows without printing the full dataset.

In [ ]:
def summarize_column(frame: pd.DataFrame, column: str) -> dict:
    series = frame[column]
    result = {
        "column": column,
        "dtype": str(series.dtype),
        "non_null_count": int(series.notna().sum()),
        "missing_count": int(series.isna().sum()),
        "missing_pct": float(series.isna().mean() * 100),
        "n_unique": int(series.nunique(dropna=True)),
        "unique_pct": float(series.nunique(dropna=True) / len(frame) * 100),
        "min": np.nan,
        "max": np.nan,
        "mean": np.nan,
        "median": np.nan,
        "std": np.nan,
    }
    if pd.api.types.is_numeric_dtype(series):
        result.update(
            {
                "min": series.min(),
                "max": series.max(),
                "mean": series.mean(),
                "median": series.median(),
                "std": series.std(),
            }
        )
    return result


column_summary = pd.DataFrame([summarize_column(train, column) for column in train.columns])

display(Markdown(f"**Dataset shape:** {train.shape[0]:,} rows × {train.shape[1]:,} columns"))
display(train.head())
display(train.sample(n=min(5, len(train)), random_state=RANDOM_STATE))
train.info()
display(column_summary)

## 5. Target / Fraud Prevalence

The target distribution describes the labels in this training data. It is not evidence of the production fraud rate and does not determine the probability of an individual claim.

In [ ]:
target_counts = train["label"].value_counts(dropna=False).rename_axis("label").rename("claim_count")
target_share = train["label"].value_counts(normalize=True, dropna=False).rename_axis("label").rename("share")
target_summary = pd.concat([target_counts, target_share], axis=1).reset_index()
fraud_prevalence = float(train["label"].mean())
fraud_count = int((train["label"] == 1).sum())
non_fraud_count = int((train["label"] == 0).sum())
imbalance_ratio = non_fraud_count / fraud_count if fraud_count else np.nan

display(target_summary)
print(f"Non-fraud claims: {non_fraud_count:,}")
print(f"Fraud claims: {fraud_count:,}")
print(f"Fraud prevalence: {fraud_prevalence:.2%}")
print(f"Non-fraud to fraud ratio: {imbalance_ratio:.3f}:1")

ax = target_counts.sort_index().plot.bar(color=["#4C78A8", "#E45756"])
ax.set_title("Training-Label Distribution")
ax.set_xlabel("Label (0 = non-fraud, 1 = fraud)")
ax.set_ylabel("Number of claims")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

display(
    Markdown(
        f"The observed fraud prevalence is **{fraud_prevalence:.2%}**. Later model evaluation should use "
        "ranking, audit-budget, and probability-quality metrics rather than relying on accuracy alone."
    )
)

## 6. Data Type and Semantic Review

Logical types are analytical classifications, not claims about clinical semantics. The official data dictionary remains authoritative.

In [ ]:
diagnosis_columns = [column for column in train.columns if column.startswith("dx2_")]
procedure_columns = [column for column in train.columns if column.startswith("proc")]
grouped_count_columns = diagnosis_columns + procedure_columns
known_categorical_columns = [
    column
    for column in ["kdkc", "dati2", "typeppk", "jkpst", "jnspelsep", "cmg", "severitylevel", "diagprimer"]
    if column in train.columns
]


def infer_logical_type(column: str) -> tuple[str, str]:
    if column == "claim_id":
        return "identifier", "Observation identifier only; prohibited from predictive use."
    if column == "label":
        return "target", "Competition fraud label."
    if column in grouped_count_columns:
        values = set(train[column].dropna().unique())
        if values.issubset({0, 1}):
            return "binary indicator", "Observed as binary; diagnosis/procedure semantics require confirmation."
        return "count-like grouped field", "Observed values exceed 1; do not assume a binary indicator."
    if column in {"umur", "los"}:
        return "numeric", "Likely age or length of stay from the brief; requires data dictionary confirmation."
    if column in known_categorical_columns:
        return "categorical code", "Code-like values are categorical; leading zeros and definitions matter."
    if pd.api.types.is_numeric_dtype(train[column]):
        return "numeric", "Numeric representation; semantic meaning requires confirmation."
    return "unknown / requires data dictionary confirmation", "No confirmed semantic interpretation is available."


semantic_records = []
for column in train.columns:
    logical_type, notes = infer_logical_type(column)
    semantic_records.append(
        {
            "column": column,
            "raw_dtype": str(train[column].dtype),
            "logical_type": logical_type,
            "n_unique": int(train[column].nunique(dropna=True)),
            "notes": notes,
        }
    )
semantic_review = pd.DataFrame(semantic_records)
categorical_columns = semantic_review.loc[
    semantic_review["logical_type"].eq("categorical code"), "column"
].tolist()
numeric_columns = semantic_review.loc[
    semantic_review["logical_type"].eq("numeric"), "column"
].tolist()

display(semantic_review)
display(
    Markdown(
        "Numeric-looking geography and administrative codes are kept as strings. Their meanings, missing-value "
        "conventions, and decision-time availability require the official data dictionary."
    )
)

## 7. Missing and Inconsistent Values

Missingness and representation issues are documented before any future cleaning. A blank, placeholder, or coding convention may have different meanings.

In [ ]:
missing_summary = (
    column_summary.loc[:, ["column", "missing_count", "missing_pct"]]
    .sort_values(["missing_pct", "column"], ascending=[False, True])
    .reset_index(drop=True)
)
display(missing_summary)

columns_with_missing = missing_summary.loc[missing_summary["missing_count"] > 0]
if columns_with_missing.empty:
    display(Markdown("No missing values were parsed from the training CSV. This does not rule out special-value encodings."))
missing_plot_data = columns_with_missing if not columns_with_missing.empty else missing_summary
ax = missing_plot_data.plot.barh(x="column", y="missing_pct", legend=False, color="#F58518", figsize=(10, 12))
ax.set_title("Missing Values by Column")
ax.set_xlabel("Missing values (%)")
ax.set_ylabel("Column")
plt.tight_layout()
plt.show()

inconsistency_records = []
for column in train.select_dtypes(include=["string", "object"]).columns:
    raw = train[column].astype("string")
    stripped = raw.str.strip()
    casefolded = stripped.str.casefold()
    non_null = raw.dropna()
    lengths = stripped.dropna().str.len()
    inconsistency_records.append(
        {
            "column": column,
            "raw_n_unique": int(raw.nunique(dropna=True)),
            "stripped_n_unique": int(stripped.nunique(dropna=True)),
            "casefolded_n_unique": int(casefolded.nunique(dropna=True)),
            "rows_changed_by_strip": int((raw != stripped).fillna(False).sum()),
            "placeholder_count": int(casefolded.isin(PLACEHOLDER_VALUES).sum()),
            "n_unique_code_lengths": int(lengths.nunique()),
            "observed_code_lengths": ", ".join(map(str, sorted(lengths.dropna().unique()))),
            "numeric_like_count": int(stripped.str.fullmatch(r"\d+").fillna(False).sum()),
        }
    )
inconsistency_summary = pd.DataFrame(inconsistency_records).sort_values(
    ["rows_changed_by_strip", "placeholder_count"], ascending=False
)
display(inconsistency_summary)
normalization_examples = []
for column in train.select_dtypes(include=["string", "object"]).columns:
    raw = train[column].astype("string")
    stripped = raw.str.strip()
    changed = pd.DataFrame({"raw_value": raw, "stripped_value": stripped}).loc[raw.ne(stripped).fillna(False)]
    if not changed.empty:
        examples = (
            changed.value_counts(dropna=False)
            .rename("count")
            .reset_index()
            .assign(column=column)
        )
        normalization_examples.append(examples)
normalization_examples = (
    pd.concat(normalization_examples, ignore_index=True)
    .sort_values("count", ascending=False)
    .head(20)
    if normalization_examples
    else pd.DataFrame(columns=["raw_value", "stripped_value", "count", "column"])
)
display(normalization_examples)
display(
    Markdown(
        "The comparison uses derived stripped values for diagnosis only. The raw values remain unchanged; any "
        "normalization for modeling should be specified and validated later."
    )
)

## 8. Duplicate and Near-Duplicate Analysis

Repeated records can indicate valid repeated claims, data duplication, limited feature resolution, or a validation risk. They are not evidence of fraud by themselves.

In [ ]:
feature_columns = [column for column in train.columns if column not in {"claim_id", "label"}]
full_duplicate_count = int(train.duplicated().sum())
duplicate_claim_id_count = int(train["claim_id"].duplicated().sum())
duplicate_feature_mask = train.duplicated(subset=feature_columns, keep=False)

feature_profiles = (
    train.groupby(feature_columns, dropna=False, observed=True)["label"]
    .agg(profile_count="size", fraud_count="sum", label_nunique="nunique")
    .reset_index(drop=True)
)
repeated_feature_profiles = feature_profiles.loc[feature_profiles["profile_count"] > 1]
mixed_label_profiles = repeated_feature_profiles.loc[repeated_feature_profiles["label_nunique"] > 1]

duplicate_summary = pd.DataFrame(
    [
        {
            "check": "Full-row duplicates",
            "count": full_duplicate_count,
            "pct_of_rows": full_duplicate_count / len(train) * 100,
        },
        {
            "check": "Duplicate claim_id values",
            "count": duplicate_claim_id_count,
            "pct_of_rows": duplicate_claim_id_count / len(train) * 100,
        },
        {
            "check": "Rows in repeated feature profiles",
            "count": int(duplicate_feature_mask.sum()),
            "pct_of_rows": float(duplicate_feature_mask.mean() * 100),
        },
        {
            "check": "Repeated feature profiles",
            "count": int(len(repeated_feature_profiles)),
            "pct_of_rows": int(len(repeated_feature_profiles)) / len(feature_profiles) * 100,
        },
        {
            "check": "Repeated profiles with mixed labels",
            "count": int(len(mixed_label_profiles)),
            "pct_of_rows": int(len(mixed_label_profiles)) / max(len(feature_profiles), 1) * 100,
        },
    ]
)
display(duplicate_summary)

feature_hash = pd.util.hash_pandas_object(train[feature_columns], index=False)
duplicate_id_audit = (
    pd.DataFrame({"claim_id": train["claim_id"], "label": train["label"], "feature_hash": feature_hash})
    .groupby("claim_id", dropna=False, observed=True)
    .agg(record_count=("label", "size"), feature_vectors=("feature_hash", "nunique"), label_values=("label", "nunique"))
    .reset_index()
)
duplicate_id_detail = duplicate_id_audit.loc[duplicate_id_audit["record_count"] > 1].sort_values(
    "record_count", ascending=False
)
display(duplicate_id_detail.head(20))

near_profile_columns = [
    column
    for column in [
        "kdkc",
        "dati2",
        "typeppk",
        "jkpst",
        "umur",
        "jnspelsep",
        "los",
        "cmg",
        "severitylevel",
        "diagprimer",
    ]
    if column in train.columns
]
near_duplicate_profiles = (
    train.groupby(near_profile_columns, dropna=False, observed=True)["label"]
    .agg(profile_count="size", fraud_count="sum")
    .assign(fraud_rate=lambda frame: frame["fraud_count"] / frame["profile_count"])
    .reset_index()
)
repeated_near_profiles = near_duplicate_profiles.loc[
    near_duplicate_profiles["profile_count"] > 1
].sort_values("profile_count", ascending=False)
display(repeated_near_profiles.head(20))

display(
    Markdown(
        f"There are **{len(mixed_label_profiles):,}** repeated full-feature profiles with mixed labels. If present, "
        "these profiles show that the available variables do not fully determine the label; they do not prove data error."
    )
)

## 9. Implausible and Unusual Observations

This section flags unusual values for review. Without confirmed definitions, it avoids calling a value impossible or removing an outlier.

In [ ]:
def numeric_summary_table(frame: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    records = []
    for column in columns:
        series = pd.to_numeric(frame[column], errors="coerce")
        quantile_values = series.quantile(NUMERIC_QUANTILES)
        record = {
            "column": column,
            "non_null_count": int(series.notna().sum()),
            "negative_count": int((series < 0).sum()),
            "zero_count": int((series == 0).sum()),
            "min": series.min(),
            "max": series.max(),
            "mean": series.mean(),
            "median": series.median(),
            "std": series.std(),
            "skewness": series.skew(),
        }
        for quantile, value in quantile_values.items():
            record[f"q_{quantile:g}"] = value
        records.append(record)
    return pd.DataFrame(records)


numeric_summary = numeric_summary_table(train, numeric_columns + grouped_count_columns)
display(numeric_summary)

unusual_numeric_summary = numeric_summary.loc[
    numeric_summary["column"].isin(["umur", "los"])
].copy()
display(unusual_numeric_summary)
display(
    Markdown(
        "`umur` and `los` are inspected because the competition brief suggests age and length of stay. Their exact "
        "definitions, permitted zero values, and valid ranges remain unconfirmed without a data dictionary."
    )
)

## 10. Categorical Cardinality Analysis

The thresholds below are EDA conventions, not domain rules. High cardinality can create sparse groups and unstable category-level estimates.

In [ ]:
cardinality_records = []
for column in categorical_columns:
    counts = train[column].value_counts(dropna=False)
    top_value = counts.index[0] if not counts.empty else pd.NA
    top_count = int(counts.iloc[0]) if not counts.empty else 0
    n_unique = int(train[column].nunique(dropna=True))
    if n_unique <= CARDINALITY_BINS[0]:
        cardinality_level = "low"
    elif n_unique <= CARDINALITY_BINS[1]:
        cardinality_level = "medium"
    elif n_unique <= CARDINALITY_BINS[2]:
        cardinality_level = "high"
    else:
        cardinality_level = "very high"
    cardinality_records.append(
        {
            "column": column,
            "n_unique": n_unique,
            "unique_pct": n_unique / len(train) * 100,
            "most_common_value": top_value,
            "most_common_count": top_count,
            "most_common_pct": top_count / len(train) * 100,
            "cardinality_level": cardinality_level,
        }
    )
categorical_cardinality = pd.DataFrame(cardinality_records).sort_values("n_unique", ascending=False)
display(categorical_cardinality)

ax = categorical_cardinality.plot.bar(x="column", y="n_unique", legend=False, color="#72B7B2")
ax.set_title("Categorical Feature Cardinality")
ax.set_xlabel("Categorical feature")
ax.set_ylabel("Unique values")
plt.xticks(rotation=50, ha="right")
plt.tight_layout()
plt.show()

rare_category_records = []
for column in categorical_columns:
    counts = train[column].value_counts(dropna=False).rename_axis("category").rename("count").reset_index()
    for threshold in [10, 50, 100]:
        rare_category_records.append(
            {
                "column": column,
                "threshold": f"count < {threshold}",
                "n_categories": int((counts["count"] < threshold).sum()),
                "claims_in_categories": int(counts.loc[counts["count"] < threshold, "count"].sum()),
            }
        )
rare_category_summary = pd.DataFrame(rare_category_records)
display(rare_category_summary)

## 11. Numeric Distribution Analysis

Plots focus on the likely operational numeric fields rather than generating dozens of unreadable charts.

In [ ]:
def plot_numeric_distribution(series: pd.Series, label: str) -> None:
    values = pd.to_numeric(series, errors="coerce").dropna()
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
    axes[0].hist(values, bins=40, color="#4C78A8", edgecolor="white")
    axes[0].set_title(f"Distribution of {label}")
    axes[0].set_xlabel(label)
    axes[0].set_ylabel("Number of claims")
    axes[1].boxplot(values, vert=False)
    axes[1].set_title(f"Boxplot of {label}")
    axes[1].set_xlabel(label)
    plt.tight_layout()
    plt.show()


for column in ["umur", "los"]:
    if column in train.columns:
        plot_numeric_distribution(train[column], column)

## 12. Diagnosis and Procedure Feature Analysis

The prefixes identify grouped diagnosis and procedure fields, but their data values must determine whether a binary interpretation is valid.

In [ ]:
def conditional_fraud_rate(frame: pd.DataFrame, mask: pd.Series, target: str = "label") -> float:
    selected = frame.loc[mask, target]
    return float(selected.mean()) if len(selected) else np.nan


def summarize_grouped_fields(frame: pd.DataFrame, columns: list[str], target: str = "label") -> pd.DataFrame:
    records = []
    for column in columns:
        series = pd.to_numeric(frame[column], errors="coerce")
        observed_values = sorted(series.dropna().unique().tolist())
        positive_mask = series.gt(0)
        records.append(
            {
                "feature": column,
                "positive_count": int(positive_mask.sum()),
                "positive_pct": float(positive_mask.mean() * 100),
                "fraud_rate_when_0": conditional_fraud_rate(frame, series.eq(0), target),
                "fraud_rate_when_1": conditional_fraud_rate(frame, series.eq(1), target),
                "fraud_rate_when_positive": conditional_fraud_rate(frame, positive_mask, target),
                "max_value": series.max(),
                "observed_values": ", ".join(map(lambda value: f"{value:g}", observed_values)),
                "is_binary": set(observed_values).issubset({0, 1}),
            }
        )
    return pd.DataFrame(records).sort_values("positive_count", ascending=False)


diagnosis_feature_summary = summarize_grouped_fields(train, diagnosis_columns)
procedure_feature_summary = summarize_grouped_fields(train, procedure_columns)
binary_feature_summary = pd.concat(
    [
        diagnosis_feature_summary.assign(feature_group="diagnosis"),
        procedure_feature_summary.assign(feature_group="procedure"),
    ],
    ignore_index=True,
)
display(diagnosis_feature_summary)
display(procedure_feature_summary)

analysis = train.assign(
    secondary_diagnosis_count=train[diagnosis_columns].sum(axis=1),
    procedure_count=train[procedure_columns].sum(axis=1),
)
non_binary_grouped_fields = binary_feature_summary.loc[~binary_feature_summary["is_binary"]]
all_zero_grouped_fields = binary_feature_summary.loc[binary_feature_summary["positive_count"].eq(0)]
aggregate_count_summary = numeric_summary_table(
    analysis,
    ["secondary_diagnosis_count", "procedure_count"],
)
display(aggregate_count_summary)
display(all_zero_grouped_fields)
display(
    Markdown(
        f"**{len(non_binary_grouped_fields):,}** grouped diagnosis/procedure fields contain values beyond 0 and 1. "
        "They are analyzed as non-negative count-like values, pending dictionary confirmation."
    )
)

def plot_count_distribution(series: pd.Series, label: str) -> None:
    counts = series.value_counts().sort_index()
    ax = counts.plot.bar(color="#54A24B")
    ax.set_title(f"Distribution of {label}")
    ax.set_xlabel(label)
    ax.set_ylabel("Number of claims")
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()


plot_count_distribution(analysis["secondary_diagnosis_count"], "Secondary diagnosis count")
plot_count_distribution(analysis["procedure_count"], "Procedure count")

## 13. Fraud Pattern Analysis

Every observed fraud-rate table includes sample size. A high rate in a small group is not treated as reliable evidence or as a causal explanation.

In [ ]:
def fraud_rate_table(
    frame: pd.DataFrame,
    feature: str,
    target: str = "label",
    min_count: int = 1,
) -> pd.DataFrame:
    grouping = frame[feature].astype("string").fillna("<MISSING>")
    grouped = (
        pd.DataFrame({"group": grouping, target: frame[target]})
        .groupby("group", dropna=False, observed=True)[target]
        .agg(claim_count="size", fraud_count="sum")
        .reset_index()
    )
    grouped["fraud_rate"] = grouped["fraud_count"] / grouped["claim_count"]
    grouped["overall_fraud_rate"] = fraud_prevalence
    grouped["fraud_rate_difference"] = grouped["fraud_rate"] - fraud_prevalence
    grouped["relative_fraud_rate"] = grouped["fraud_rate"] / fraud_prevalence if fraud_prevalence else np.nan
    return grouped.loc[grouped["claim_count"] >= min_count].sort_values(
        ["claim_count", "fraud_rate"], ascending=[False, False]
    ).reset_index(drop=True)


def plot_fraud_rate(table: pd.DataFrame, title: str, max_groups: int = 15) -> None:
    selected = table.sort_values("claim_count", ascending=False).head(max_groups).sort_values("fraud_rate")
    ax = selected.plot.barh(x="group", y="fraud_rate", legend=False, color="#E45756")
    ax.axvline(fraud_prevalence, color="black", linestyle="--", label="Overall fraud prevalence")
    ax.set_title(title)
    ax.set_xlabel("Observed fraud rate")
    ax.set_ylabel("Group")
    ax.legend()
    plt.tight_layout()
    plt.show()


if "umur" in analysis.columns:
    age_bins = [-np.inf, 17, 39, 59, np.inf]
    age_labels = ["0-17", "18-39", "40-59", "60+"]
    analysis["age_group"] = pd.cut(analysis["umur"], bins=age_bins, labels=age_labels)
    fraud_by_age = fraud_rate_table(analysis, "age_group")
    display(fraud_by_age)
    plot_fraud_rate(fraud_by_age, "Observed Fraud Rate by Provisional Age Group", max_groups=4)
else:
    fraud_by_age = pd.DataFrame()

gender_column = "jkpst" if "jkpst" in analysis.columns else None
if gender_column:
    fraud_by_gender = fraud_rate_table(analysis, gender_column)
    display(fraud_by_gender)
    plot_fraud_rate(fraud_by_gender, "Observed Fraud Rate by Raw Demographic Category")
else:
    fraud_by_gender = pd.DataFrame()

fraud_by_feature = {}
selected_categorical_features = [
    column
    for column in ["kdkc", "dati2", "typeppk", "jnspelsep", "cmg", "severitylevel", "diagprimer"]
    if column in analysis.columns
]
for column in selected_categorical_features:
    table = fraud_rate_table(analysis, column, min_count=MIN_GROUP_SIZE)
    fraud_by_feature[column] = table
    display(Markdown(f"### {column}: groups with at least {MIN_GROUP_SIZE:,} claims"))
    display(table.head(20))

for column in ["typeppk", "cmg", "severitylevel", "diagprimer"]:
    if column in fraud_by_feature:
        plot_fraud_rate(fraud_by_feature[column], f"Observed Fraud Rate by {column}")

fraud_by_diagnosis_count = fraud_rate_table(analysis, "secondary_diagnosis_count")
fraud_by_procedure_count = fraud_rate_table(analysis, "procedure_count")
display(fraud_by_diagnosis_count)
display(fraud_by_procedure_count)
plot_fraud_rate(fraud_by_diagnosis_count, "Observed Fraud Rate by Secondary Diagnosis Count")
plot_fraud_rate(fraud_by_procedure_count, "Observed Fraud Rate by Procedure Count")

if "los" in analysis.columns:
    analysis["los_band"] = pd.cut(
        analysis["los"],
        bins=[-np.inf, 0, 3, 7, 30, np.inf],
        labels=["0", "1-3", "4-7", "8-30", "31+"],
    )
    fraud_by_los_band = fraud_rate_table(analysis, "los_band")
    display(fraud_by_los_band)
    plot_fraud_rate(fraud_by_los_band, "Observed Fraud Rate by Length-of-Stay Band", max_groups=5)
else:
    fraud_by_los_band = pd.DataFrame()

display(
    Markdown(
        "All rates above are observed label associations. They do not demonstrate that a demographic, clinical, "
        "geographic, or facility characteristic causes fraud."
    )
)

## 14. Potential Proxy Variable Analysis

Potential proxies may encode related information. Association strength is descriptive and requires domain review; it does not establish a protected characteristic or a fairness conclusion.

In [ ]:
def cramers_v(left: pd.Series, right: pd.Series) -> float:
    contingency = pd.crosstab(left.astype("string"), right.astype("string"), dropna=False)
    if min(contingency.shape) < 2:
        return np.nan
    chi2 = stats.chi2_contingency(contingency, correction=False)[0]
    n = contingency.to_numpy().sum()
    denominator = n * min(contingency.shape[0] - 1, contingency.shape[1] - 1)
    return float(np.sqrt(chi2 / denominator)) if denominator else np.nan


proxy_candidates = [
    ("kdkc", "dati2", "Geography codes may encode administrative hierarchy."),
    ("kdkc", "typeppk", "Facility type may be concentrated by geography."),
    ("dati2", "typeppk", "Facility type may be concentrated by geography."),
    ("jkpst", "age_group", "Clinical use and demographics may be associated."),
    ("jkpst", "diagprimer", "Diagnosis patterns may vary across raw demographic categories."),
    ("age_group", "diagprimer", "Diagnosis patterns may vary across provisional age groups."),
    ("age_group", "jnspelsep", "Service type may vary across provisional age groups."),
    ("typeppk", "jnspelsep", "Facility and service types may be structurally related."),
]
proxy_records = []
for left, right, rationale in proxy_candidates:
    if left in analysis.columns and right in analysis.columns:
        proxy_records.append(
            {
                "feature_a": left,
                "feature_b": right,
                "cramers_v": cramers_v(analysis[left], analysis[right]),
                "interpretation": rationale,
                "review_status": "Potential proxy relationship; requires domain review.",
            }
        )
proxy_analysis = pd.DataFrame(proxy_records).sort_values("cramers_v", ascending=False)
display(proxy_analysis)
display(
    Markdown(
        "Cramér’s V measures categorical association strength, not causality, fairness, or predictive validity. "
        "No sensitive information is inferred and no external data are used."
    )
)

## 15. Information Leakage Audit

Leakage is information unavailable at the audit-prioritization decision. Statistical separation is a review signal only; strong prediction can be valid, leaked, or an artifact of the dataset.

In [ ]:
LEAKAGE_PURITY_THRESHOLD = 0.98

category_leakage_records = []
for feature, table in fraud_by_feature.items():
    substantial = table.loc[table["claim_count"] >= MIN_GROUP_SIZE].copy()
    substantial["near_pure_label_rate"] = substantial["fraud_rate"].le(1 - LEAKAGE_PURITY_THRESHOLD) | substantial[
        "fraud_rate"
    ].ge(LEAKAGE_PURITY_THRESHOLD)
    for row in substantial.itertuples(index=False):
        category_leakage_records.append(
            {
                "feature": feature,
                "category": row.group,
                "claim_count": row.claim_count,
                "fraud_count": row.fraud_count,
                "fraud_rate": row.fraud_rate,
                "near_pure_label_rate": row.near_pure_label_rate,
            }
        )
suspicious_categories = pd.DataFrame(category_leakage_records)
if not suspicious_categories.empty:
    suspicious_categories = suspicious_categories.sort_values(
        ["near_pure_label_rate", "claim_count", "fraud_rate"], ascending=[False, False, False]
    )
display(suspicious_categories.head(30))

leakage_binary_scan = binary_feature_summary.loc[
    :, ["feature", "feature_group", "positive_count", "positive_pct", "fraud_rate_when_0", "fraud_rate_when_positive"]
].copy()
leakage_binary_scan["absolute_rate_difference"] = (
    leakage_binary_scan["fraud_rate_when_positive"] - leakage_binary_scan["fraud_rate_when_0"]
).abs()
leakage_binary_scan = leakage_binary_scan.sort_values("absolute_rate_difference", ascending=False)
display(leakage_binary_scan.head(30))

leakage_records = []
for row in semantic_review.itertuples(index=False):
    if row.column == "claim_id":
        availability = "not a predictor"
        potential_leakage = "yes"
        reason = "Observation identifier; explicitly prohibited from predictive use and feature engineering."
        requires_confirmation = False
    elif row.column == "label":
        availability = "target only"
        potential_leakage = "not applicable"
        reason = "Training target; unavailable for evaluation claims."
        requires_confirmation = False
    else:
        availability = "unknown"
        potential_leakage = "requires review"
        reason = "Decision-time availability is not confirmed because no official data dictionary is present."
        requires_confirmation = True
    leakage_records.append(
        {
            "feature": row.column,
            "logical_type": row.logical_type,
            "available_before_audit_decision": availability,
            "potential_leakage": potential_leakage,
            "reason": reason,
            "requires_dictionary_confirmation": requires_confirmation,
        }
    )
leakage_review = pd.DataFrame(leakage_records)
display(leakage_review)
display(
    Markdown(
        "Near-perfect labels with substantial sample sizes should be reviewed for post-decision meaning. This notebook "
        "does not automatically remove any field or label a predictive association as leakage."
    )
)

## 16. Data Limitations

Only observed issues and explicitly missing documentation are listed below. These limitations should shape later cleaning, validation, and modeling decisions.

In [ ]:
limitation_records = [
    {
        "limitation": "No official data dictionary is available in the repository.",
        "evidence": "Feature definitions, permitted values, missing-value conventions, and decision-time availability are unconfirmed.",
        "impact": "Semantic assumptions and leakage review remain provisional.",
    },
    {
        "limitation": "Categorical representation requires review.",
        "evidence": f"{int(inconsistency_summary['rows_changed_by_strip'].sum()):,} string-field values change after trimming.",
        "impact": "Raw-category fragmentation can affect category counts and model encoding.",
    },
    {
        "limitation": "Diagnosis and procedure field semantics are unresolved.",
        "evidence": f"{len(non_binary_grouped_fields):,} grouped fields contain values beyond 0 and 1.",
        "impact": "Binary-indicator treatment would discard observed count information or misstate meaning.",
    },
    {
        "limitation": "High-cardinality categorical codes are present.",
        "evidence": f"{int((categorical_cardinality['cardinality_level'].isin(['high', 'very high'])).sum()):,} categorical fields meet the high or very-high EDA threshold.",
        "impact": "Rare groups can make estimates unstable and increase overfitting risk.",
    },
]
if len(mixed_label_profiles):
    limitation_records.append(
        {
            "limitation": "Some repeated feature profiles have conflicting labels.",
            "evidence": f"{len(mixed_label_profiles):,} repeated full-feature profiles contain both label values.",
            "impact": "The measured features may not fully resolve the label; probabilistic predictions remain appropriate.",
        }
    )
if not columns_with_missing.empty:
    limitation_records.append(
        {
            "limitation": "Parsed missing values are present.",
            "evidence": f"{len(columns_with_missing):,} columns contain missing values.",
            "impact": "Missingness mechanisms and appropriate handling need later validation.",
        }
    )
data_limitations = pd.DataFrame(limitation_records)
display(data_limitations)

## 17. Key Findings

Findings below are generated from the current training data and are framed as evidence for later work, not as causal claims.

In [ ]:
findings_records = [
    {
        "Finding": "Observed training-label prevalence",
        "Evidence": f"Fraud prevalence = {fraud_prevalence:.2%} ({fraud_count:,} of {len(train):,} claims).",
        "Why It Matters": "Target composition informs metric baselines and probability interpretation.",
        "Modeling Implication": "Evaluate ranking and calibration with out-of-fold predictions.",
    },
    {
        "Finding": "Code-like categorical fields vary in cardinality",
        "Evidence": f"Largest non-identifier categorical cardinality = {int(categorical_cardinality['n_unique'].max()):,}.",
        "Why It Matters": "Sparse categories can create unstable fraud-rate estimates.",
        "Modeling Implication": "Use categorical-aware preprocessing and validate rare-category handling.",
    },
    {
        "Finding": "Grouped diagnosis and procedure values are not uniformly binary",
        "Evidence": f"{len(non_binary_grouped_fields):,} fields have observed values above 1.",
        "Why It Matters": "A binary-only interpretation would be inconsistent with the data.",
        "Modeling Implication": "Treat them as count-like pending dictionary confirmation.",
    },
    {
        "Finding": "Decision-time feature availability is not documented",
        "Evidence": "No official data dictionary is available in the repository.",
        "Why It Matters": "Post-decision fields could introduce leakage.",
        "Modeling Implication": "Complete a semantic availability review before model fitting.",
    },
]
if int(inconsistency_summary["rows_changed_by_strip"].sum()):
    findings_records.append(
        {
            "Finding": "Some string values differ after trimming",
            "Evidence": f"{int(inconsistency_summary['rows_changed_by_strip'].sum()):,} values change in a derived strip comparison.",
            "Why It Matters": "Raw representation can fragment otherwise related categories.",
            "Modeling Implication": "Compare explicit normalization strategies within leakage-safe validation.",
        }
    )
if len(mixed_label_profiles):
    findings_records.append(
        {
            "Finding": "Repeated profiles can have mixed labels",
            "Evidence": f"{len(mixed_label_profiles):,} repeated full-feature profiles contain both classes.",
            "Why It Matters": "Features do not perfectly determine outcomes.",
            "Modeling Implication": "Favor calibrated probabilities over deterministic rules.",
        }
    )
key_findings = pd.DataFrame(findings_records)
display(key_findings)

## 18. Recommendations for Modeling

These are next-step investigations for Task B, not preprocessing performed in this notebook.

In [ ]:
modeling_recommendations = pd.DataFrame(
    [
        {
            "recommendation": "Keep administrative and clinical codes categorical unless the data dictionary states otherwise.",
            "evidence": "Leading-zero codes and categorical cardinality review.",
        },
        {
            "recommendation": "Preserve raw strings and test explicitly documented normalization for whitespace or placeholder variants.",
            "evidence": "Derived inconsistency comparison identifies representation differences.",
        },
        {
            "recommendation": "Treat diagnosis and procedure fields as count-like until their definitions are confirmed.",
            "evidence": "Observed values above 1 in grouped fields.",
        },
        {
            "recommendation": "Use leakage-safe out-of-fold validation and inspect grouping or time variables if documentation becomes available.",
            "evidence": "Repeated profiles and unconfirmed decision-time availability.",
        },
        {
            "recommendation": "Evaluate calibrated probabilistic models with audit-budget and ranking metrics, not accuracy alone.",
            "evidence": "Competition objective prioritizes top-5% audit allocation and probability quality.",
        },
        {
            "recommendation": "Review potential proxy variables and subgroup audit exposure before operational use.",
            "evidence": "Observed categorical associations require domain and fairness review.",
        },
    ]
)
display(modeling_recommendations)
display(
    Markdown(
        "The next modeling stage should retain human oversight: the model ranks audit priority, while investigators "
        "make fraud determinations."
    )
)